In [1]:
import torch
from torch import tensor

In [2]:
n = 10
x = torch.randint(1,10, size=(n,3), dtype=torch.float32, requires_grad=True) # {n} data with 3 dimensions
y = torch.randint(0,2, size=(n,1), dtype=torch.float32, requires_grad=True)

x, y

(tensor([[4., 8., 8.],
         [2., 1., 3.],
         [6., 9., 3.],
         [5., 5., 7.],
         [4., 8., 3.],
         [6., 5., 8.],
         [7., 9., 6.],
         [7., 2., 2.],
         [7., 4., 5.],
         [3., 4., 4.]], requires_grad=True),
 tensor([[1.],
         [1.],
         [0.],
         [1.],
         [0.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.]], requires_grad=True))

In [79]:
def check_grads(grad1, grad2):
    assert grad1.shape == grad2.shape, "shape mismatch"
    assert torch.allclose(grad1, grad2), "grads not close"

In [ ]:
n = 10

x = torch.randint(1,10, size=(n,3), dtype=torch.float32) # {n} data with 3 dimensions
y = torch.randint(0,2, size=(n,1), dtype=torch.int)

w = torch.randn(size=(3,2), requires_grad=True)
b = torch.randn(size=(1,2), requires_grad=True)
a = x @ w; a.retain_grad() #done
logits = a + b; logits.retain_grad() #done

#log_sum_exp = torch.logsumexp(logits, dim=1, keepdim=True) #https://www.perplexity.ai/search/05ad9f81-20e9-4b52-8a37-6e2c1eaa1852
logits_exp = logits.exp(); logits_exp.retain_grad() #done
logits_exp_sum = torch.sum(logits_exp, dim=1, keepdim=True); logits_exp_sum.retain_grad() #done
log_sum_exp = torch.log(logits_exp_sum); log_sum_exp.retain_grad() #done

log_probs = logits - log_sum_exp; log_probs.retain_grad() #done
nll = - log_probs[torch.arange(0, n), y.squeeze()]; nll.retain_grad() #done
loss = nll.mean(dim=0); loss.retain_grad() #done

loss.backward()

with torch.no_grad():
    d_loss = 1.0
    d_nll = torch.ones_like(nll)/n * d_loss; check_grads(d_nll, nll.grad)
    
    d_temp = torch.zeros_like(log_probs)
    d_temp[torch.arange(0, n), y.squeeze()] = 1.0
    d_log_probs = -1.0 * d_temp * d_nll.unsqueeze(dim=1); check_grads(d_log_probs, log_probs.grad)
    
    d_logits = 1.0 * d_log_probs
    d_log_sum_exp = -1.0 * d_log_probs.sum(dim=1, keepdim=True); check_grads(d_log_sum_exp, log_sum_exp.grad)
    
    d_logits_exp_sum = logits_exp_sum ** -1 * d_log_sum_exp; check_grads(d_logits_exp_sum, logits_exp_sum.grad)
    d_logits_exp = d_logits_exp_sum.expand_as(logits_exp); check_grads(d_logits_exp, logits_exp.grad)
    d_logits += logits_exp * d_logits_exp; check_grads(d_logits, logits.grad)
    
    d_b = 1.0 * d_logits.sum(dim=0, keepdim=True); check_grads(d_b, b.grad)
    d_a = d_logits; check_grads(d_a, a.grad)
    
    d_w =  x.T @ d_a; check_grads(d_w, w.grad)